<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Adding Other Users' SSH Keys to a Slice

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**Welcome!** This notebook demonstrates how to add additional SSH public keys to your FABRIC slice so that other team members can access the nodes. By default, only your own SSH key is installed on slice VMs. Adding extra keys enables collaborative experiments where multiple researchers need direct SSH access to the same nodes.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Add **extra SSH public keys** to a slice at creation time using the `extra_ssh_keys` parameter
2. Verify that additional keys are properly installed in the VM's `authorized_keys` file
3. Understand how FABRIC handles SSH key management for collaborative experiments

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Be familiar with creating slices (see [Hello, FABRIC](../hello_fabric/hello_fabric.ipynb))
3. Obtain the **SSH public keys** from the users you want to grant access to

**Tip:** Users must share their **public** key (e.g., `id_ecdsa.pub`), never the private key. Public keys typically start with `ssh-rsa`, `ecdsa-sha2-nistp256`, or `ssh-ed25519`.

</div>

## Background: SSH Key Management in FABRIC

When you create a FABRIC slice, your SSH public key (configured during environment setup) is automatically installed in the `~/.ssh/authorized_keys` file on every VM in the slice. This allows you to SSH into your nodes through the bastion host.

For **collaborative experiments**, you may want other team members to also have SSH access. FABRIC supports this through the `extra_ssh_keys` parameter on `slice.submit()`. These additional keys are installed alongside your own key during slice provisioning.


<div class="fab-danger">

**Security note:** Only add keys from trusted collaborators. Anyone with a key in `authorized_keys` has full SSH access to the node, including root access via `sudo`.

</div>

## What We're Building

In this notebook we will create a single compute node with default resources.

<img src="./figs/slice_topology.png" width="40%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Create the Slice with Extra SSH Keys

We create a slice with a single node and provide two additional SSH public keys at submission time. In a real experiment, you would replace these example keys with the actual public keys from your collaborators.

<div class="fab-danger">

**Warning:** The keys shown below are **examples only**. Do **not** use these keys in real experiments. Replace them with the actual public keys from your collaborators.

</div>

In [ ]:
# Create a new slice
slice = fablib.new_slice(name="MySlice with extra SSH keys")

# Add a single node with default settings
node = slice.add_node(name="Node1")

# Define the extra SSH public keys to install on the node.
# Each key is a full public key string (the contents of an .pub file).
# Replace these example keys with real public keys from your collaborators.
extra_ssh_keys=['ecdsa-sha2-nistp256 AAAAE2VjZHNhLXNoYTItbmlzdHAyNTYAAAAIbmlzdHAyNTYAAABBBI1scIhcI0VLxiTZlI4Zt1rxUFfU3nISDZhDm2fk4CZbdMAsOec/5Oq2UjN7yu7hibsVprysMMPoxXc7OfjQfXk= userkey1',
                'ecdsa-sha2-nistp256 AAAAE2VjZHNhLXNoYTItbmlzdHAyNTYAAAAIbmlzdHAyNTYAAABBBH+0ua+cFUwR23rcuTRHB7IpbSof5c3qQuQlKb6vaEgqLuNe+9EM7nJKwQVJqr2glSN+qZeXfXSRqkcqLQLZ4uA= userkey2']

# Submit the slice with the extra keys.
# The 'extra_ssh_keys' parameter is a list of public key strings
# that will be added to authorized_keys on ALL nodes in the slice.
slice.submit(extra_ssh_keys=extra_ssh_keys);

## Step 3: Inspect the Slice

Verify that the slice and node are active.

In [ ]:
# Display slice-level information (state, expiration, project)
slice.show();

## Step 4: Verify the Keys Are Installed

Let's SSH into the node and check the `authorized_keys` file to confirm that both our key and the extra keys were installed.

In [ ]:
# Get the node reference
node = slice.get_node('Node1')

# Display the contents of the authorized_keys file.
# You should see your own key plus the two extra keys.
command = 'cat .ssh/authorized_keys'

stdout, stderr = node.execute(command)

<div class="fab-success">

**Success!** If you see three or more keys in the output above (your key plus the two extra keys), the additional SSH keys were properly installed. Your collaborators can now SSH into this node using their private keys.

</div>

## Step 5: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- leaving slices running unnecessarily prevents other researchers from using those resources.

</div>

In [ ]:
# Delete the slice and release all resources
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| Extra keys not in `authorized_keys` | Keys not passed correctly to `submit()` | Verify `extra_ssh_keys` is a list of valid public key strings |
| Collaborator cannot SSH in | Collaborator using wrong private key | Ensure they use the private key matching the public key you added |
| `Permission denied (publickey)` | Bastion host not configured for collaborator | Collaborator must have their own FABRIC bastion access configured |
| Key format error | Invalid public key string | Public keys must be complete single-line strings starting with key type (e.g., `ssh-rsa`, `ecdsa-sha2-nistp256`) |
| Only your key appears | `extra_ssh_keys` parameter was empty or not passed | Check that the list is not empty and is passed as a named parameter to `submit()` |
| Slice creation fails | Unrelated provisioning issue | Check error messages; try a different site |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `slice.add_node(name)` | Add a compute node to the slice | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `slice.submit(extra_ssh_keys)` | Submit the slice with additional SSH keys | [submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) |
| `slice.show()` | Display slice attributes | [show](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.show) |
| `slice.get_node(name)` | Get a specific node object by name | [get_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_node) |
| `node.execute(command)` | Execute a shell command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

Now that you know how to add SSH keys for collaboration, explore these related notebooks:

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **Hello, FABRIC** | [hello_fabric](../hello_fabric/hello_fabric.ipynb) | Create your first slice and run commands |
| **Customizing Nodes** | [customizing_nodes](../customizing_nodes/customizing_nodes.ipynb) | Set site, cores, RAM, disk, and OS image |
| **Upload and Execute** | [upload_and_execute](../upload_and_execute/upload_and_execute.ipynb) | Upload scripts, execute them, and download results |
| **Execute Commands** | [execute_commands](../ssh_to_nodes/execute_commands.ipynb) | More ways to run commands and capture output |
| **Networking** | [FABnet IPv4 (auto)](../create_l3network_fabnet_ipv4/create_l3network_fabnet_ipv4_auto.ipynb) | Connect nodes across sites with Layer 3 networking |